# Camada Gold — Data Marts e KPIs de Negócio

Este notebook constrói os Data Marts consolidados e responde às perguntas-chave
de negócio usando `display()` para exibição direta no Databricks.

**Nenhuma biblioteca de visualização externa é necessária.**

Projetos:
1. **Visão Comercial e Volume de Produtos** — receita por categoria e rankings de produtos
2. **Satisfação de Clientes e Qualidade de Parceiros** — avaliações por categoria, vendedor e estado

## 1 — Imports e Configurações

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

CATALOG       = "medalhao"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD   = "gold"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA_GOLD}")

print("Catálogo e schema configurados:", CATALOG, ">", SCHEMA_GOLD)

## 2 — Função Auxiliar de Escrita Gold
Todas as gravações na Gold usam modo `overwrite` para garantir idempotência.

In [0]:
def salvar_gold(df, nome_tabela: str):
    """
    Grava um DataFrame como tabela Delta na camada Gold.
    Modo overwrite garante que reexecutar o notebook sempre produza resultado atualizado.
    """
    tabela_completa = f"{CATALOG}.{SCHEMA_GOLD}.{nome_tabela}"
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela_completa)
    )
    count = spark.table(tabela_completa).count()
    print(f"✅ {tabela_completa} → {count:,} registros")

## 3 — Leitura das Tabelas Silver
Carrega todas as tabelas silver necessárias para construção dos Data Marts.

In [0]:
df_pedido_total   = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.fat_pedido_total")
df_itens_pedidos  = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.fat_itens_pedidos")
df_produtos       = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.dim_produtos")
df_vendedores     = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.dim_vendedores")
df_avaliacoes     = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.fat_avaliacoes_pedidos")
df_categoria_trad = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.dim_categoria_produtos_traducao")
df_pedidos        = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.fat_pedidos")

print("Tabelas Silver carregadas com sucesso.")

---
# 🛒 Projeto 1: Visão Comercial e Volume de Produtos

## 4 — gold.fat_vendas_comercial

Granularidade: Ano × Mês × Categoria de Produto.

Métricas:
- `total_pedidos`: contagem distinta de pedidos
- `qtd_itens_vendidos`: contagem absoluta de itens
- `receita_total_brl`: soma do valor pago em BRL
- `receita_total_usd`: soma do valor pago em USD
- `ticket_medio_brl`: receita_total_brl / total_pedidos

In [0]:
df_base_comercial = (
    df_itens_pedidos
    .join(df_produtos.select("id_produto", "categoria_produto"), on="id_produto", how="left")
    .join(
        df_pedido_total.select(
            "id_pedido", "valor_total_pago_brl", "valor_total_pago_usd", "data_pedido"
        ),
        on="id_pedido",
        how="left"
    )
    .withColumn("ano_venda", F.year(F.col("data_pedido")))
    .withColumn("mes_venda",  F.month(F.col("data_pedido")))
)

df_fat_vendas_comercial = (
    df_base_comercial
    .groupBy("ano_venda", "mes_venda", "categoria_produto")
    .agg(
        F.countDistinct("id_pedido").alias("total_pedidos"),
        F.count("id_item").alias("qtd_itens_vendidos"),
        F.round(F.sum("valor_total_pago_brl"), 2).alias("receita_total_brl"),
        F.round(F.sum("valor_total_pago_usd"), 2).alias("receita_total_usd"),
    )
    .withColumn(
        "ticket_medio_brl",
        F.round(F.col("receita_total_brl") / F.col("total_pedidos"), 2)
    )
    .orderBy("ano_venda", "mes_venda", "categoria_produto")
)

salvar_gold(df_fat_vendas_comercial, "fat_vendas_comercial")
display(df_fat_vendas_comercial)

## 5 — Rankings Comerciais (display)
Top 5 produtos mais e menos vendidos em todo o histórico.

In [0]:

df_ranking_base = (
    df_itens_pedidos
    .join(
        df_produtos.select("id_produto", "nome_produto", "categoria_produto"),
        on="id_produto",
        how="left"
    )
    .groupBy("id_produto", "nome_produto", "categoria_produto")
    .agg(F.count("id_item").alias("quantidade_vendida"))
)

print("📈 Top 5 Produtos MAIS Vendidos:")
display(
    df_ranking_base
    .select("nome_produto", "categoria_produto", "quantidade_vendida")
    .orderBy(F.desc("quantidade_vendida"))
    .limit(5)
)

In [0]:

print("📉 Top 5 Produtos MENOS Vendidos:")
display(
    df_ranking_base
    .select("nome_produto", "categoria_produto", "quantidade_vendida")
    .orderBy(F.asc("quantidade_vendida"))
    .limit(5)
)

---
# ⭐ Projeto 2: Satisfação de Clientes e Qualidade de Parceiros

## 6 — gold.fat_avaliacoes_clientes

Granularidade: Categoria do Produto × Nome do Vendedor × Estado do Vendedor.

Métricas:
- `total_avaliacoes`: contagem absoluta
- `avaliacao_media`: média das notas
- `total_avaliacoes_positivas`: notas >= 4
- `total_avaliacoes_negativas`: notas <= 2
- `percentual_satisfacao`: positivas / total * 100

In [0]:

df_base_avaliacoes = (
    df_avaliacoes
    .join(df_itens_pedidos.select("id_pedido", "id_produto", "id_vendedor"),
          on="id_pedido", how="left")
    .join(df_produtos.select("id_produto", "categoria_produto"),
          on="id_produto", how="left")
    .join(df_vendedores.select("id_vendedor", "nome_vendedor", "estado"),
          on="id_vendedor", how="left")
)

df_fat_avaliacoes_clientes = (
    df_base_avaliacoes
    .groupBy("categoria_produto", "nome_vendedor", "estado")
    .agg(
        F.count("id_avaliacao").alias("total_avaliacoes"),
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
        F.sum(F.when(F.col("nota_avaliacao") >= 4, 1).otherwise(0))
          .alias("total_avaliacoes_positivas"),
        F.sum(F.when(F.col("nota_avaliacao") <= 2, 1).otherwise(0))
          .alias("total_avaliacoes_negativas"),
    )
    .withColumn(
        "percentual_satisfacao",
        F.round(
            (F.col("total_avaliacoes_positivas") / F.col("total_avaliacoes")) * 100,
            2
        )
    )
    .orderBy(F.desc("total_avaliacoes"))
)

salvar_gold(df_fat_avaliacoes_clientes, "fat_avaliacoes_clientes")
display(df_fat_avaliacoes_clientes)

## 7 — Rankings de Qualidade (display)
Critério de ordenação composto:
1. Nota (maior/menor)
2. Volume de avaliações como desempate (decrescente)

Mínimo de 10 avaliações para garantir representatividade estatística.

In [0]:

df_ranking_produto_completo = (
    df_base_avaliacoes
    .join(df_produtos.select("id_produto", "nome_produto"), on="id_produto", how="left")
    .groupBy("id_produto", "nome_produto", "categoria_produto")
    .agg(
        F.count("id_avaliacao").alias("total_avaliacoes"),
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
    )
    .filter(F.col("total_avaliacoes") >= 10)
    .select("nome_produto", "categoria_produto", "avaliacao_media", "total_avaliacoes")
)


In [0]:

print("⭐ Produto MAIS bem avaliado:")
display(
    df_ranking_produto_completo
    .orderBy(F.desc("avaliacao_media"), F.desc("total_avaliacoes"))
    .limit(1)
)

In [0]:

print("💔 Produto MENOS bem avaliado:")
display(
    df_ranking_produto_completo
    .orderBy(F.asc("avaliacao_media"), F.desc("total_avaliacoes"))
    .limit(1)
)

In [0]:

df_ranking_vendedor = (
    df_base_avaliacoes
    .groupBy("id_vendedor", "nome_vendedor", "estado")
    .agg(
        F.count("id_avaliacao").alias("total_avaliacoes"),
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
    )
    .filter(F.col("total_avaliacoes") >= 10)
    .select("nome_vendedor", "estado", "avaliacao_media", "total_avaliacoes")
)

print("🏆 Vendedor MAIS bem avaliado:")
display(
    df_ranking_vendedor
    .orderBy(F.desc("avaliacao_media"), F.desc("total_avaliacoes"))
    .limit(1)
)

In [0]:
print("⚠️ Vendedor MENOS bem avaliado:")
display(
    df_ranking_vendedor
    .orderBy(F.asc("avaliacao_media"), F.desc("total_avaliacoes"))
    .limit(1)
)

## 8 — Validação Final

In [0]:
print("=== Tabelas na camada Gold ===")
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA_GOLD}"))